# 模型保存与加载练习

本模块练习 `state_dict`、训练检查点、恢复优化器状态、跨设备加载和推理一致性。运行练习时会在当前目录创建 checkpoints 文件夹。

In [ ]:
from pathlib import Path
import torch
from torch import nn

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
torch.manual_seed(42)

## 练习 1：检查 state_dict ⭐

创建 `Linear(3,2)`，取得它的 state_dict，并确认其中权重与偏置的名称和形状。

In [ ]:
model = nn.Linear(3, 2)

# TODO
state = None
parameter_names = None
weight_shape = None
bias_shape = None

assert parameter_names == {"weight", "bias"}
assert weight_shape == (2, 3)
assert bias_shape == (2,)
print("✅ 练习 1 通过")

## 练习 2：只保存和加载模型参数 ⭐⭐

保存 model 的 state_dict。创建结构相同的新模型并加载参数，使两个模型对同一输入产生完全相同的输出。

In [ ]:
model_path = checkpoint_dir / "linear_state_dict.pt"
test_input = torch.randn(4, 3)
expected_output = model(test_input).detach()

# TODO：保存 state_dict
loaded_model = nn.Linear(3, 2)
# TODO：从 model_path 加载并交给 loaded_model
loaded_model.eval()

with torch.no_grad():
    loaded_output = loaded_model(test_input)
assert model_path.exists()
assert torch.equal(loaded_output, expected_output)
print("✅ 练习 2 通过")

## 练习 3：保存完整训练检查点 ⭐⭐

创建包含 epoch、模型参数、优化器参数和 loss 的字典并保存。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
checkpoint_path = checkpoint_dir / "training_checkpoint.pt"

# 先执行一次训练步骤，让优化器产生内部状态
train_x = torch.randn(8, 3)
train_y = torch.randn(8, 2)
loss = nn.MSELoss()(model(train_x), train_y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# TODO
checkpoint = None
# 保存 checkpoint

assert checkpoint_path.exists()
assert set(checkpoint) == {"epoch", "model_state_dict", "optimizer_state_dict", "loss"}
assert checkpoint["epoch"] == 5
print("✅ 练习 3 通过")

## 练习 4：恢复训练状态 ⭐⭐⭐

创建新模型和新优化器，从检查点恢复二者，并取出下一轮 epoch 和上一次 loss。

In [ ]:
resumed_model = nn.Linear(3, 2)
resumed_optimizer = torch.optim.Adam(resumed_model.parameters(), lr=0.001)

# TODO：加载 checkpoint，恢复模型与优化器
next_epoch = None
previous_loss = None

for old_parameter, new_parameter in zip(model.parameters(), resumed_model.parameters()):
    assert torch.equal(old_parameter, new_parameter)
assert len(resumed_optimizer.state) > 0
assert next_epoch == 6
assert isinstance(previous_loss, float)
print("✅ 练习 4 通过")

## 练习 5：跨设备加载 ⭐⭐

用 map_location 把模型参数明确加载到 CPU，并验证所有参数都位于 CPU。

In [ ]:
cpu_model = nn.Linear(3, 2)

# TODO
cpu_state = None
# 加载到 cpu_model

assert all(parameter.device.type == "cpu" for parameter in cpu_model.parameters())
with torch.no_grad():
    assert torch.equal(cpu_model(test_input), expected_output)
print("✅ 练习 5 通过")

## 练习 6：保存后进行可复现推理 ⭐⭐⭐

补全预测函数。函数应切换评估模式、关闭梯度，并返回 softmax 概率。

In [ ]:
classifier = nn.Sequential(nn.Linear(3, 5), nn.ReLU(), nn.Dropout(0.5), nn.Linear(5, 2))
classifier_path = checkpoint_dir / "classifier.pt"
torch.save(classifier.state_dict(), classifier_path)

reloaded_classifier = nn.Sequential(nn.Linear(3, 5), nn.ReLU(), nn.Dropout(0.5), nn.Linear(5, 2))
reloaded_classifier.load_state_dict(torch.load(classifier_path, map_location="cpu", weights_only=True))


def predict_probabilities(model, inputs):
    # TODO
    pass


inputs = torch.randn(6, 3)
probabilities_1 = predict_probabilities(reloaded_classifier, inputs)
probabilities_2 = predict_probabilities(reloaded_classifier, inputs)
assert torch.equal(probabilities_1, probabilities_2)
assert torch.allclose(probabilities_1.sum(dim=1), torch.ones(6))
assert not probabilities_1.requires_grad
print("✅ 练习 6 通过")

## 过关标准

你应该知道普通推理只需保存 state_dict，而继续训练通常还需保存优化器状态、epoch 等信息；加载外部文件时只使用可信检查点。